In [1]:
# ════════════════════════════════════════════════════════════════════
# Cellule 1 — Setup
# ════════════════════════════════════════════════════════════════════
import os, subprocess

os.environ['CUDA_VISIBLE_DEVICES']    = '0'
os.environ['PYTORCH_ALLOC_CONF']      = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
os.environ['WANDB_DISABLED']          = 'true'

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Pas de GPU detecte')

!pip install -q evaluate rouge-score bert-score sentencepiece accelerate
print('Installation OK')

Tue May 12 00:25:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ════════════════════════════════════════════════════════════════════
# Cellule 2 — Chargement et nettoyage du dataset
# ════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import re
import json
import glob
from sklearn.model_selection import train_test_split

# ── Trouver le fichier Excel ──────────────────────────────────────────
excel_files = glob.glob('/kaggle/input/**/*.xlsx', recursive=True)
EXCEL_PATH  = excel_files[0] if excel_files else None
print(f'Dataset : {EXCEL_PATH}')

SHEETS = {
    'Sport رياضة':           'sport',
    'Politique سياسة':       'politique',
    'Economie اقتصاد':       'economie',
    'Culture ثقافة':         'culture',
    'Religion دين':          'religion',
    'Technologie تكنولوجيا': 'technologie',
    'Autre أخرى': 'autre'
}

# ── Charger toutes les feuilles ───────────────────────────────────────
dfs = []
for sheet, domain in SHEETS.items():
    try:
        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
        df['domain'] = domain
        dfs.append(df)
        print(f'  {sheet:<30} → {len(df)} articles')
    except Exception as e:
        print(f'  {sheet} → ERREUR : {e}')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nTotal chargé : {len(df_all)} articles')

Dataset : /kaggle/input/datasets/fakhetwalid/tunisum-dataset22k/dataset_COMPLET.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Sport رياضة                    → 5049 articles


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Politique سياسة                → 7966 articles


...

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Religion دين                   → 1587 articles


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Technologie تكنولوجيا          → 819 articles
  Autre أخرى                     → 1860 articles

Total chargé : 22650 articles


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# ════════════════════════════════════════════════════════════════════
# Cellule 3 — Prétraitement  (Texte_Complet → Résumé/Description)
# ════════════════════════════════════════════════════════════════════
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def is_bad_pair(desc, text):
    desc = desc.strip()
    text = text.strip()
    
    if not desc or not text:
        return False
    
    # 🔴 1. Comparer avec le début du texte uniquement
    text_start = text[:len(desc)]
    
    # 🔴 2. Similarité sur le début
    if similarity(desc, text_start) > 0.9:
        return True
    
    # 🔴 3. Inclusion approximative (sécurité)
    if desc in text:
        return True
    
    return False


def normalize_arabic(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'[\u0617-\u061A\u064B-\u065F\u0670]', '', text)
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def detect_arabic_ratio(text):
    if not text: return 0.0
    arabic = len(re.findall(r'[\u0600-\u06FF]', text))
    total  = len(text.replace(' ', ''))
    return arabic / total if total > 0 else 0.0

# ── Normalisation ──────────────────────────────────────────────────
print('Normalisation en cours...')
df_all['text_norm'] = df_all['Texte Complet'].apply(normalize_arabic)
df_all['desc_norm'] = df_all['Description'].apply(normalize_arabic)

# ── Filtrage qualité ───────────────────────────────────────────────
n0 = len(df_all)

# 1. Supprimer nulls
df = df_all.dropna(subset=['Texte Complet', 'Description']).copy()
df = df[df['Texte Complet'].str.strip() != '']
df = df[df['Description'].str.strip()   != '']
print(f'[1] Nulls/vides            : -{n0 - len(df)}')

# 2. Supprimer doublons URL
n = len(df)
df = df.drop_duplicates(subset=['Lien'], keep='first')
print(f'[2] Doublons URL           : -{n - len(df)}')

# 3. Supprimer texte_complet == description (copie exacte)
n = len(df)
df = df[df['text_norm'].str.strip() != df['desc_norm'].str.strip()]
print(f'[3] Texte = Description    : -{n - len(df)}')



# 4. Supprimer description contenue entièrement dans texte (leading substring)
#    Cas fréquent : description = premiers 2 phrases du texte sans reformulation
n = len(df)
#df = df[~df.apply(
#    lambda r: r['desc_norm'].strip() in r['text_norm'].strip(), axis=1
#)]
df = df[df.apply(
    lambda r: similarity(r['desc_norm'].strip(), r['text_norm'].strip()) < 0.7,
    axis=1
)]
#df = df[~df.apply(lambda r: is_bad_pair(r['desc_norm'], r['text_norm']), axis=1)]

print(f'[4] Description ⊂ Texte    : -{n - len(df)}')

# 5. Filtrage longueur
#    - texte_complet : min 300 chars (articles trop courts = peu de contenu)
#                      max 12000 chars (éviter docs hors-domaine)
#    - description   : min 30 chars, max 600 chars (résumé court attendu)
n = len(df)
df = df[df['text_norm'].str.len().between(300, 12000)]
df = df[df['desc_norm'].str.len().between(30,  600  )]
print(f'[5] Longueur hors seuils   : -{n - len(df)}')

# 6. Ratio de compression minimal (description doit être plus courte que texte)
#    On exige que la description fasse au max 70% de la longueur du texte
n = len(df)
df = df[df['desc_norm'].str.len() / df['text_norm'].str.len() <= 0.70]
print(f'[6] Ratio compression <30% : -{n - len(df)}')

# 7. Filtrage ratio arabe
n = len(df)
df = df[df['text_norm'].apply(detect_arabic_ratio) >= 0.5]
df = df[df['desc_norm'].apply(detect_arabic_ratio) >= 0.5]
print(f'[7] Ratio arabe < 50%      : -{n - len(df)}')

df = df.reset_index(drop=True)

# ── Stats finales ──────────────────────────────────────────────────
print(f'\nDataset final : {len(df)} paires')
print(f'Texte  moy    : {df["text_norm"].str.len().mean():.0f} chars | '
      f'{df["text_norm"].apply(lambda x: len(x.split())).mean():.0f} mots')
print(f'Desc   moy    : {df["desc_norm"].str.len().mean():.0f} chars | '
      f'{df["desc_norm"].apply(lambda x: len(x.split())).mean():.0f} mots')
ratio_moy = (df['desc_norm'].str.len() / df['text_norm'].str.len()).mean()
print(f'Ratio  moy    : {ratio_moy:.2%}  (compression moyenne)')

# ── Préparation pour le fine-tuning ───────────────────────────────
# La cellule suivante attend des colonnes 'text' et 'summary'
df['text']    = df['text_norm']
df['summary'] = df['desc_norm']

Normalisation en cours...
[1] Nulls/vides            : -7327
[2] Doublons URL           : -23
[3] Texte = Description    : -354
[4] Description ⊂ Texte    : -5557
[5] Longueur hors seuils   : -1058
[6] Ratio compression <30% : -300
[7] Ratio arabe < 50%      : -1

Dataset final : 8030 paires
Texte  moy    : 3871 chars | 604 mots
Desc   moy    : 246 chars | 40 mots
Ratio  moy    : 17.30%  (compression moyenne)


In [4]:
df.head()

,#,Source,Langue,Domaine,Date de publication,Titre,Description,Lien,Texte Complet,domain,text_norm,desc_norm,text,summary
0,141,Leaders.com.tn,AR,Sport / رياضة,18/03/2026 17:28,Ooredoo تونس تحتفل بعيد الفطر بمبادرة خاصة لفا...,بمناسبة عيد الفطر المبارك، نظّمت Ooredoo تونس ...,https://ar.leaders.com.tn/article/7514-ooredoo...,بمناسبة عيد الفطر المبارك، نظّمت Ooredoo تونس ...,sport,بمناسبة عيد الفطر المبارك، نظمت Ooredoo تونس م...,بمناسبة عيد الفطر المبارك، نظمت Ooredoo تونس م...,بمناسبة عيد الفطر المبارك، نظمت Ooredoo تونس م...,بمناسبة عيد الفطر المبارك، نظمت Ooredoo تونس م...
1,142,Leaders.com.tn,AR,Sport / رياضة,16/03/2026 12:02,تابعوا بصور حصرية الفوز الثمين للترجي الرياضي ...,شهد ملعب حمادي العقربي برادس مساء الأحد فصلاً ...,https://ar.leaders.com.tn/article/7510-%D8%AF%...,شهد ملعب حمادي العقربي برادس مساء الأحد فصلاً ...,sport,شهد ملعب حمادي العقربي برادس مساء الاحد فصلا ج...,شهد ملعب حمادي العقربي برادس مساء الاحد فصلا ج...,شهد ملعب حمادي العقربي برادس مساء الاحد فصلا ج...,شهد ملعب حمادي العقربي برادس مساء الاحد فصلا ج...
2,143,Leaders.com.tn,AR,Sport / رياضة,13/03/2026 12:34,"البنك العربي لتونس (ATB) و""Visa"" يطلقان مسابقة...","يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi...",https://ar.leaders.com.tn/article/7507-%D8%A7%...,"يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi...",sport,"يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi...","يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi...","يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi...","يعود البنك العربي لتونس (ATB)، بالتعاون مع ""Vi..."
3,146,Jawhara FM - Politique,AR,Sport / رياضة,20/03/2026 11:05,عبد الباري عطوان: 'تونس من الأصوات القليلة الت...,أشاد الكاتب والصحفي الفلسطيني عبد الباري عطوان...,https://www.jawharafm.net/ar/article/عبد-البار...,الاستقبال >أخبار >سياسة سياسة 2026/03/20 11:04...,sport,الاستقبال >اخبار >سياسة سياسة 2026/03/20 11:04...,اشاد الكاتب والصحفي الفلسطيني عبد الباري عطوان...,الاستقبال >اخبار >سياسة سياسة 2026/03/20 11:04...,اشاد الكاتب والصحفي الفلسطيني عبد الباري عطوان...
4,147,Jawhara FM - Politique,AR,Sport / رياضة,05/03/2026 09:41,عطوان: 'أمريكا ليست معصومة من الهزيمة في حربها...,قال الكاتب والصحفي الفلسطيني، عبد الباري عطوان...,https://www.jawharafm.net/ar/article/عطوان-أمر...,الاستقبال >أخبار >سياسة سياسة 2026/03/05 09:35...,sport,الاستقبال >اخبار >سياسة سياسة 2026/03/05 09:35...,قال الكاتب والصحفي الفلسطيني، عبد الباري عطوان...,الاستقبال >اخبار >سياسة سياسة 2026/03/05 09:35...,قال الكاتب والصحفي الفلسطيني، عبد الباري عطوان...


In [5]:
# ════════════════════════════════════════════════════════════════════
# Cellule 4 — Création des splits et sauvegarde
#             (Texte_Complet → Résumé/Description)
# ════════════════════════════════════════════════════════════════════
import os, json
import pandas as pd
from sklearn.model_selection import train_test_split

OUTPUT_DIR = '/kaggle/working/tunisum_summarization'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Split stratifié 80/10/10
df_train, df_temp = train_test_split(
    df, test_size=0.20,
    stratify=df['domain'],
    random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50,
    stratify=df_temp['domain'],
    random_state=42
)
print(f'Train : {len(df_train)} | Val : {len(df_val)} | Test : {len(df_test)}')

def save_json(data, path):
    records = []
    for _, row in data.iterrows():
        records.append({
            'id':          str(row.name),
            'text':        row['text_norm'],     # input  : texte complet
            'summary':     row['desc_norm'],     # output : description/résumé
            'text_raw':    str(row['Texte Complet']),
            'summary_raw': str(row['Description']),
            'domain':      row['domain'],
            'source':      str(row['Source']) if 'source' in row else '',
            'url':         str(row['Lien']) if pd.notna(row['Lien']) else '',
            'text_len':    len(row['text_norm'].split()),
            'summary_len': len(row['desc_norm'].split()),
        })
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f'Sauvegarde : {path} → {len(records)} exemples')

save_json(df_train, f'{OUTPUT_DIR}/train.json')
save_json(df_val,   f'{OUTPUT_DIR}/val.json')
save_json(df_test,  f'{OUTPUT_DIR}/test.json')

# Statistiques par domaine
print(f'\nDistribution par domaine :')
for domain, group in df.groupby('domain'):
    pct = len(group) / len(df) * 100
    print(f'  {domain:<15} : {len(group):>4} ({pct:.1f}%)')

# Statistiques de longueur par split
print(f'\nLongueurs moyennes (en mots) :')
for nom, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    t = split['text_norm'].apply(lambda x: len(x.split())).mean()
    s = split['desc_norm'].apply(lambda x: len(x.split())).mean()
    print(f'  {nom} — texte: {t:.0f} mots | résumé: {s:.0f} mots')

Train : 6424 | Val : 803 | Test : 803
Sauvegarde : /kaggle/working/tunisum_summarization/train.json → 6424 exemples
Sauvegarde : /kaggle/working/tunisum_summarization/val.json → 803 exemples
Sauvegarde : /kaggle/working/tunisum_summarization/test.json → 803 exemples

Distribution par domaine :
  autre           :  533 (6.6%)
  culture         :  616 (7.7%)
  economie        : 1397 (17.4%)
  politique       : 3218 (40.1%)
  religion        :  624 (7.8%)
  sport           : 1434 (17.9%)
  technologie     :  208 (2.6%)

Longueurs moyennes (en mots) :
  Train — texte: 602 mots | résumé: 40 mots
  Val — texte: 599 mots | résumé: 40 mots
  Test — texte: 628 mots | résumé: 40 mots


In [6]:
%%writefile /kaggle/working/fine_tune.py

# ════════════════════════════════════════════════════════════════════
# Cellule 5 — Script fine_tune.py (commun à tous les modèles)
# ════════════════════════════════════════════════════════════════════

import json, argparse, os, sys
import numpy as np

MODELS = {
    'arat5': {
        'name':   'UBC-NLP/AraT5-base-title-generation',
        'prefix': 'summarize: ',
        'note':   'AraT5 base - titre generation'
    },
    'arat5v2': {
        'name':   'UBC-NLP/AraT5v2-base-1024',
        'prefix': 'summarize: ',
        'note':   'AraT5 v2 - sequences longues'
    },
    'mt5': {
        'name':   'google/mt5-base',
        'prefix': 'summarize: ',
        'note':   'mT5 base - baseline multilingue'
    },
}

def load_data(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_rouge_fn():
    for key in list(sys.modules.keys()):
        if key == 'evaluate' or key.startswith('evaluate.'):
            del sys.modules[key]
    try:
        import evaluate as ev
        if hasattr(ev, 'load'):
            rouge = ev.load('rouge')
            print('HuggingFace evaluate OK')
            def fn(preds, refs):
                r = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
                return {k: round(v * 100, 4) for k, v in r.items()}
            return fn
    except Exception as e:
        print(f'Fallback rouge_score ({e})')
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    def fn(preds, refs):
        s1, s2, sL = [], [], []
        for p, r in zip(preds, refs):
            s = sc.score(r, p)
            s1.append(s['rouge1'].fmeasure)
            s2.append(s['rouge2'].fmeasure)
            sL.append(s['rougeL'].fmeasure)
        return {
            'rouge1': round(np.mean(s1) * 100, 4),
            'rouge2': round(np.mean(s2) * 100, 4),
            'rougeL': round(np.mean(sL) * 100, 4),
        }
    return fn

def main():
    parser = argparse.ArgumentParser(description='TuniSum Fine-tuning')
    parser.add_argument('--model',      default='arat5', choices=MODELS.keys())
    parser.add_argument('--train',      required=True)
    parser.add_argument('--val',        required=True)
    parser.add_argument('--output',     required=True)
    parser.add_argument('--epochs',     type=int,   default=10)
    parser.add_argument('--batch-size', type=int,   default=4)
    parser.add_argument('--lr',         type=float, default=5e-4)
    parser.add_argument('--max-input',  type=int,   default=512)
    parser.add_argument('--max-target', type=int,   default=64)
    args = parser.parse_args()

    from transformers import (
        AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig,
        Seq2SeqTrainer, Seq2SeqTrainingArguments,
        DataCollatorForSeq2Seq, EarlyStoppingCallback,
    )
    from datasets import Dataset

    cfg = MODELS[args.model]
    print(f'\nModele    : {cfg["name"]}')
    print(f'Note      : {cfg["note"]}')
    print(f'Epochs    : {args.epochs}')
    print(f'Batch     : {args.batch_size} (effectif: {args.batch_size * max(1, 32 // args.batch_size)})')
    print(f'LR        : {args.lr}')
    print(f'Max input : {args.max_input} tokens')
    print(f'Max target: {args.max_target} tokens\n')

    # Fix tie_word_embeddings warning
    config = AutoConfig.from_pretrained(cfg['name'])
    config.tie_word_embeddings = False
    tokenizer = AutoTokenizer.from_pretrained(cfg['name'])
    model     = AutoModelForSeq2SeqLM.from_pretrained(cfg['name'], config=config)

    compute_rouge = get_rouge_fn()

    train_data = load_data(args.train)
    val_data   = load_data(args.val)
    print(f'Train : {len(train_data)} | Val : {len(val_data)}')

    prefix = cfg['prefix']

    def tokenize_fn(example):
        inp = tokenizer(
            prefix + example['text'],
            max_length=args.max_input,
            truncation=True,
            padding='max_length'
        )
        tgt = tokenizer(
            example['summary'],
            max_length=args.max_target,
            truncation=True,
            padding='max_length'
        )
        labels = [(l if l != tokenizer.pad_token_id else -100)
                  for l in tgt['input_ids']]
        return {
            'input_ids':      inp['input_ids'],
            'attention_mask': inp['attention_mask'],
            'labels':         labels,
        }

    print('Tokenisation...')
    col_names = list(Dataset.from_list(train_data[:1]).column_names)
    train_ds  = Dataset.from_list(train_data).map(tokenize_fn, remove_columns=col_names)
    val_ds    = Dataset.from_list(val_data).map(tokenize_fn,   remove_columns=col_names)

    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        dp = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
        dl = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        result = compute_rouge(dp, dl)
        result['gen_len'] = round(float(np.mean([len(p.split()) for p in dp])), 2)
        return result

    grad_accum = max(1, 32 // args.batch_size)

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=grad_accum,
        fp16=True,
        predict_with_generate=True,
        generation_max_length=args.max_target,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='rougeL',
        greater_is_better=True,
        learning_rate=args.lr,
        warmup_steps=200,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        logging_steps=50,
        report_to='none',
        save_total_limit=2,
        seed=42,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(
            tokenizer, model=model, label_pad_token_id=-100
        ),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('Fine-tuning en cours...')
    result = trainer.train()
    trainer.save_model(args.output)
    tokenizer.save_pretrained(args.output)

    print(f'\nModele sauvegarde : {args.output}')
    print(f'Loss finale       : {result.training_loss:.4f}')

if __name__ == '__main__':
    main()

Writing /kaggle/working/fine_tune.py


In [7]:
# ════════════════════════════════════════════════════════════════════
# Cellule 6 — Fine-tuning AraT5 (modèle principal)
# ════════════════════════════════════════════════════════════════════
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

MODELS_DIR = f'{OUTPUT_DIR}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/fine_tune.py \
    --model      arat5 \
    --train      {OUTPUT_DIR}/train.json \
    --val        {OUTPUT_DIR}/val.json \
    --output     {MODELS_DIR}/arat5 \
    --epochs     10 \
    --batch-size 4 \
    --lr         5e-4 \
    --max-input  512 \
    --max-target 256

VRAM libre : 15.53 GB

Modele    : UBC-NLP/AraT5-base-title-generation
Note      : AraT5 base - titre generation
Epochs    : 10
Batch     : 4 (effectif: 32)
LR        : 0.0005
Max input : 512 tokens
Max target: 256 tokens

config.json: 100%|██████████████████████████████| 687/687 [00:00<00:00, 295kB/s]
tokenizer_config.json: 100%|██████████████████| 81.0/81.0 [00:00<00:00, 474kB/s]
spiece.model: 100%|████████████████████████| 2.44M/2.44M [00:01<00:00, 2.35MB/s]
special_tokens_map.json: 100%|████████████████| 98.0/98.0 [00:00<00:00, 652kB/s]
pytorch_model.bin:  59%|███████████▊        | 671M/1.13G [00:07<00:04, 95.7MB/s]
model.safetensors:   0%|                            | 0.00/1.13G [00:01<?, ?B/s]
Loading weights:   0%|                                  | 0/284 [00:00<?, ?it/s]
Loading weights:   0%| | 1/284 [00:00<00:00, 8192.00it/s, Materializing param=de
Loading weights:   0%| | 1/284 [00:00<00:00, 2874.78it/s, Materializing param=de
Loading weights:   1%| | 2/284 [00:00<00:00, 260

In [8]:
# ── Test rapide du modèle fine-tuné ──────────────────────────────────────
import torch
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = f'{OUTPUT_DIR}/models/arat5'
#MODEL_PATH = f'{MODEL_DIR}/arat5'
# Charger le modèle
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
print(f"✅ Modèle chargé sur {device}")

# Charger quelques exemples du test set
with open(f'{OUTPUT_DIR}/test.json', encoding='utf-8') as f:
    test = json.load(f)

# Fonction de génération
def generate_summary(text, max_input=512, max_target=64):
    inputs = tokenizer(
        "summarize: " + text,
        max_length=max_input,
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_target,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Tester sur 10 exemples variés (2 par domaine)
from collections import defaultdict
by_domain = defaultdict(list)
for ex in test:
    by_domain[ex['domain']].append(ex)

print("\n" + "="*70)
print("  RÉSULTATS DU MODÈLE AraT5 fine-tuné")
print("="*70)

for domain, examples in sorted(by_domain.items()):
    print(f"\n【 {domain.upper()} 】")
    for ex in examples[:5]:
        generated = generate_summary(ex['text'])
        print(f"  Source    : {ex['text'][:120]}...")
        print(f"  Référence : {ex['summary']}")
        print(f"  Généré    : {generated}")
        print()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Modèle chargé sur cuda

  RÉSULTATS DU MODÈLE AraT5 fine-tuné

【 AUTRE 】
  Source    : الاستقبال >اخبار >وطنية وطنية2026/05/05 11:12 Tweeter نفذت وحدات الحماية المدنية خلال ال24 ساعة الفارطة 600 تدخلا في عدة...
  Référence : نفذت وحدات الحماية المدنية خلال ال24 ساعة الفارطة 600 تدخلا في عدة مجالات، وفق ما ورد في بلاغ للمتحدث باسم الادارة العامة للحماية المدنية اصدره اليوم الثلاثاء.
  Généré    : نفذت وحدات الحماية المدنية خلال ال24 ساعة الفارطة 600 تدخلا في عدة مجالات، وفق ما ورد في بلاغ للمتحدث باسم الادارة العامة للحماية المدنية اصدره اليوم الثلاثاء.

  Source    : Stories 40 خبر نبض الملاعب مشاهدة RT STORIES شاهد.. محمد صلاح يلهو مع ابنتيه اضغط للمزيد Save post Share Facebook X VK.c...
  Référence : تستغرق الرحلة الي المريخ حاليا ما بين 7 الي 10 اشهر. لكن باحثا برازيليا تمكن من العثور علي طريق مختصر يمكن ان يخفض هذه المدة الي اقل من النصف.
  Généré    : افادت وكالة "بلومبرغ" نقلا عن مصدر في البنتاغون بان الولايات المتحدة ستسحب نحو 5000 جندي امريكي من المانيا.

  Source    : Storie

In [9]:
from rouge_score import rouge_scorer
import json, torch
from collections import defaultdict

with open(f'{OUTPUT_DIR}/test.json', encoding='utf-8') as f:
    test = json.load(f)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
s1, s2, sL = [], [], []
by_domain = defaultdict(lambda: {'s1':[], 's2':[], 'sL':[]})

print(f"Évaluation sur {len(test)} exemples...")
for i, ex in enumerate(test):
    generated = generate_summary(ex['text'])
    s = scorer.score(ex['summary'], generated)
    s1.append(s['rouge1'].fmeasure)
    s2.append(s['rouge2'].fmeasure)
    sL.append(s['rougeL'].fmeasure)
    by_domain[ex['domain']]['s1'].append(s['rouge1'].fmeasure)
    by_domain[ex['domain']]['s2'].append(s['rouge2'].fmeasure)
    by_domain[ex['domain']]['sL'].append(s['rougeL'].fmeasure)
    if i % 50 == 0:
        print(f"  {i}/{len(test)}...")

import numpy as np
print(f"\n{'='*60}")
print(f"  RÉSULTATS FINAUX — AraT5 fine-tuné (lr=5e-4)")
print(f"{'='*60}")
print(f"  ROUGE-1 : {np.mean(s1)*100:.2f}  (précédent : 18.04)")
print(f"  ROUGE-2 : {np.mean(s2)*100:.2f}  (précédent :  5.28)")
print(f"  ROUGE-L : {np.mean(sL)*100:.2f}  (précédent : 17.79)")
print(f"\n  Par domaine :")
print(f"  {'Domaine':<15} {'ROUGE-1':>9} {'ROUGE-2':>9} {'ROUGE-L':>9}")
print(f"  {'-'*45}")
for domain, scores in sorted(by_domain.items()):
    print(f"  {domain:<15} {np.mean(scores['s1'])*100:>9.2f} "
          f"{np.mean(scores['s2'])*100:>9.2f} "
          f"{np.mean(scores['sL'])*100:>9.2f}")
print(f"{'='*60}")
print(f"\n  LaTeX :")
print(f"  AraT5-TuniSum & {np.mean(s1)*100:.2f} & "
      f"{np.mean(s2)*100:.2f} & {np.mean(sL)*100:.2f} \\\\")

Évaluation sur 803 exemples...
  0/803...
  50/803...
  100/803...
  150/803...
  200/803...
  250/803...
  300/803...
  350/803...
  400/803...
  450/803...
  500/803...
  550/803...
  600/803...
  650/803...
  700/803...
  750/803...
  800/803...

  RÉSULTATS FINAUX — AraT5 fine-tuné (lr=5e-4)
  ROUGE-1 : 31.61  (précédent : 18.04)
  ROUGE-2 : 21.60  (précédent :  5.28)
  ROUGE-L : 31.61  (précédent : 17.79)

  Par domaine :
  Domaine           ROUGE-1   ROUGE-2   ROUGE-L
  ---------------------------------------------
  autre               16.46     14.58     16.46
  culture             41.91     30.76     41.91
  economie            32.72     20.71     32.72
  politique           21.88     12.63     21.88
  religion            30.87     22.08     30.87
  sport               55.26     41.52     55.26
  technologie         23.42     19.43     23.42

  LaTeX :
  AraT5-TuniSum & 31.61 & 21.60 & 31.61 \\


In [10]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import json, torch
from collections import defaultdict
import numpy as np

# Chargement dataset
with open(f'{OUTPUT_DIR}/test.json', encoding='utf-8') as f:
    test = json.load(f)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

s1, s2, sL = [], [], []
by_domain = defaultdict(lambda: {'s1':[], 's2':[], 'sL':[]})

# Listes pour BERTScore
all_preds = []
all_refs  = []

print(f"Évaluation sur {len(test)} exemples...")
for i, ex in enumerate(test):
    generated = generate_summary(ex['text'])

    # ── ROUGE ──────────────────────────────────────────────────────
    s = scorer.score(generated, ex['summary'])
    s1.append(s['rouge1'].fmeasure)
    s2.append(s['rouge2'].fmeasure)
    sL.append(s['rougeL'].fmeasure)
    by_domain[ex['domain']]['s1'].append(s['rouge1'].fmeasure)
    by_domain[ex['domain']]['s2'].append(s['rouge2'].fmeasure)
    by_domain[ex['domain']]['sL'].append(s['rougeL'].fmeasure)

    # ── Collecter pour BERTScore ────────────────────────────────────
    all_preds.append(generated)
    all_refs.append(ex['summary'])

    if i % 50 == 0:
        print(f"  {i}/{len(test)}...")

# ── Calcul BERTScore (après la boucle) ───────────────────────────────
print("\nCalcul BERTScore (peut prendre 2-3 minutes)...")
P, R, F1 = bert_score(
    all_preds,
    all_refs,
    lang='ar',              # ← langue arabe
    model_type='bert-base-multilingual-cased',  # ← modèle multilingue
    verbose=False,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
bs_p  = round(P.mean().item()  * 100, 2)
bs_r  = round(R.mean().item()  * 100, 2)
bs_f1 = round(F1.mean().item() * 100, 2)

# ── Résultats ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  RÉSULTATS FINAUX")
print(f"{'='*60}")
print(f"  ROUGE-1      : {np.mean(s1)*100:.2f}")
print(f"  ROUGE-2      : {np.mean(s2)*100:.2f}")
print(f"  ROUGE-L      : {np.mean(sL)*100:.2f}")
print(f"  BERTScore-P  : {bs_p}")
print(f"  BERTScore-R  : {bs_r}")
print(f"  BERTScore-F1 : {bs_f1}  ← métrique principale")
print(f"\n  Par domaine :")
print(f"  {'Domaine':<15} {'ROUGE-1':>9} {'ROUGE-2':>9} {'ROUGE-L':>9}")
print(f"  {'-'*45}")
for domain, scores in sorted(by_domain.items()):
    print(f"  {domain:<15} {np.mean(scores['s1'])*100:>9.2f} "
          f"{np.mean(scores['s2'])*100:>9.2f} "
          f"{np.mean(scores['sL'])*100:>9.2f}")
print(f"{'='*60}")

# ── LaTeX ─────────────────────────────────────────────────────────────
print(f"\n  LaTeX :")
print(f"  Model & {np.mean(s1)*100:.2f} & {np.mean(s2)*100:.2f} & "
      f"{np.mean(sL)*100:.2f} & {bs_f1} \\\\")

Évaluation sur 803 exemples...
  0/803...
  50/803...
  100/803...
  150/803...
  200/803...
  250/803...
  300/803...
  350/803...
  400/803...
  450/803...
  500/803...
  550/803...
  600/803...
  650/803...
  700/803...
  750/803...
  800/803...

Calcul BERTScore (peut prendre 2-3 minutes)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  RÉSULTATS FINAUX
  ROUGE-1      : 31.61
  ROUGE-2      : 21.60
  ROUGE-L      : 31.61
  BERTScore-P  : 85.39
  BERTScore-R  : 84.43
  BERTScore-F1 : 84.87  ← métrique principale

  Par domaine :
  Domaine           ROUGE-1   ROUGE-2   ROUGE-L
  ---------------------------------------------
  autre               16.46     14.58     16.46
  culture             41.91     30.76     41.91
  economie            32.72     20.71     32.72
  politique           21.88     12.63     21.88
  religion            30.87     22.08     30.87
  sport               55.26     41.52     55.26
  technologie         23.42     19.43     23.42

  LaTeX :
  Model & 31.61 & 21.60 & 31.61 & 84.87 \\


In [11]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import json, numpy as np, torch

with open(f'{OUTPUT_DIR}/test.json', encoding='utf-8') as f:
    test = json.load(f)

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
s1, s2, sL, preds, refs = [], [], [], [], []

for ex in test:
    lead = ' '.join(ex['text'].split()[:10])
    s = scorer.score(lead, ex['summary'])
    s1.append(s['rouge1'].fmeasure)
    s2.append(s['rouge2'].fmeasure)
    sL.append(s['rougeL'].fmeasure)
    preds.append(lead)
    refs.append(ex['summary'])

P, R, F1 = bert_score(preds, refs, lang='ar',
    model_type='bert-base-multilingual-cased',
    verbose=False, device='cuda')

print(f'Lead-10 & {np.mean(s1)*100:.2f} & {np.mean(s2)*100:.2f} & {np.mean(sL)*100:.2f} & {F1.mean().item()*100:.2f} \\\\')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lead-10 & 5.79 & 1.47 & 5.54 & 68.02 \\
